## 데이터 살펴보기

In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("train.csv", encoding="cp949")
test= pd.read_csv("test.csv", encoding="cp949")

train.head()

,sentence,label
0,오늘은 누구랑 같이 밥을 먹어야 할까?,불안
1,난 남들 다 받는 과외를 못 받아서 공부를 못하는 것 같아.,슬픔
2,오늘 친구가 몸이 너무 안 좋아서 친구 잘못으로 내가 대신 혼났어.,상처
3,우리 주위에는 불치병으로 인하여 어려운 분들이 많은 것 같아.,상처
4,아이들이 사춘기가 되니 예전처럼 나에게 먼저 말을 걸지 않고 자꾸 나를 피하려고 해.,당황


In [3]:
label_tags = train['label'].unique()
label_tags

array(['불안', '슬픔', '상처', '당황', '분노', '기쁨'], dtype=object)

In [4]:
print(train.iloc[:7].to_markdown())

|    | sentence                                                                          | label   |
|---:|:----------------------------------------------------------------------------------|:--------|
|  0 | 오늘은 누구랑 같이 밥을 먹어야 할까?                                              | 불안    |
|  1 | 난 남들 다 받는 과외를 못 받아서 공부를 못하는 것 같아.                           | 슬픔    |
|  2 | 오늘 친구가 몸이 너무 안 좋아서 친구 잘못으로 내가 대신 혼났어.                   | 상처    |
|  3 | 우리 주위에는 불치병으로 인하여 어려운 분들이 많은 것 같아.                       | 상처    |
|  4 | 아이들이 사춘기가 되니 예전처럼 나에게 먼저 말을 걸지 않고 자꾸 나를 피하려고 해. | 당황    |
|  5 | 딸이 결혼을 안 하려고 해서 집사람이 화가 났어.                                    | 분노    |
|  6 | 학원 친구 중에 나보다 뛰어난 애가 눈에 밟혀서 화가 나.                            | 상처    |


In [5]:
print(test.iloc[:7].to_markdown())

|    | sentence                                                                                             | label   |
|---:|:-----------------------------------------------------------------------------------------------------|:--------|
|  0 | 수험생인 아들과 기분 전환 겸 집 앞 산을 등산하다가 아들이 발목을 접질려서 너무 마음 아파.            | 당황    |
|  1 | 내 장례식만큼은 내가 원하는 대로 해 달라고 말했는데 자식들은 들은 체도 안 해. 속상해.                | 상처    |
|  2 | 친구가 지난해에 이혼을 했어. 양육비를 월 오십만 원씩 주기로 하고 아이는 부인이 키우기로 했다는 거야. | 당황    |
|  3 | 학교에 늦어서 엄마 출근길에 나 좀 데려다 달라고 했는데 엄마가 화가 났을까?                           | 분노    |
|  4 | 가족들이 집으로 빨리 돌아오라고 했는데 늦게 가서 걱정만 시켰어.                                      | 슬픔    |
|  5 | 건강에 회의적이야.                                                                                   | 불안    |
|  6 | 은퇴하고 아픈 곳이 없지만 돈이 없어.                                                                 | 당황    |


## BERT 파생모델 TEST

In [3]:
import torch

input_ids = torch.LongTensor([[31, 51, 99], [15, 5, 0]])
attention_mask = torch.LongTensor([[1, 1, 1], [1, 1, 0]])
token_type_ids = torch.LongTensor([[0, 0, 1], [0, 1, 0]])

In [7]:
from transformers import BertTokenizerFast, BertModel
tokenizer_bert = BertTokenizerFast.from_pretrained("kykim/bert-kor-base")
model_bert = BertModel.from_pretrained("kykim/bert-kor-base")

Some weights of the model checkpoint at kykim/bert-kor-base were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [8]:
output = model_bert(input_ids, attention_mask, token_type_ids)
print(output.keys())

seq_output = output.last_hidden_state
pooler_output = output.pooler_output

print(seq_output.shape)
print(pooler_output.shape)

odict_keys(['last_hidden_state', 'pooler_output'])
torch.Size([2, 3, 768])
torch.Size([2, 768])


In [9]:
text = ["목사님 주보랑 ppt(설교부분 제외) 보내드려요~", "문장이 주어지는 경우가 있다."]

token = tokenizer_bert.tokenize(text)
token_ids = tokenizer_bert.convert_tokens_to_ids(token)
print(token)
print(token_ids)

['목사', '##님', '주', '##보', '##랑', 'pp', '##t', '(', '설', '##교', '##부분', '제외', ')', '보내', '##드려요', '~', '문장', '##이', '주어', '##지는', '경우가', '있다', '.']
[35714, 8403, 6165, 8032, 8202, 25151, 8086, 2010, 4964, 8033, 14194, 15729, 2011, 14396, 16980, 2070, 31622, 8018, 20808, 14029, 15218, 13984, 2016]


In [10]:
token_ids = tokenizer_bert.encode(text, max_length=128, padding='max_length')

print(token_ids)

[2, 35714, 8403, 6165, 8032, 8202, 25151, 8086, 2010, 4964, 8033, 14194, 15729, 2011, 14396, 16980, 2070, 3, 31622, 8018, 20808, 14029, 15218, 13984, 2016, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [11]:
"""
https://misconstructed.tistory.com/80

- text : encoding 될 문장 (sequence, batch). str, List[str], List[List[str]] 을 입력으로 받을 수 있고, 이미 tokenize 된 경우(list of tokens) is_split_into_words=True 로 지정해줘야 한다.

- text_pair : text와 동일. 한 개의 문장이 아니라 sentence pair를 입력으로 주는 경우인 듯.

- add_special_tokens (True) : encoding을 할 때 special token을 추가할지 여부 

- padding (False) : True로 지정한 경우 batch 내에서 가장 긴 길이에 맞춰서 padding을 한다. False인 경우 padding을 하지 않고 encoding을 한다. 'max_length'로 지정하면 모델이 입력으로 받을 수 있는 최대의 길이에 맞춰서 padding을 한다. 

- truncation (False): True 인 경우 모델이 입력으로 받을 수 있는 최대의 길이에 맞춰서 Truncation을 한다. "only_first"인 경우 sentence pair로 입력이 주어지는 경우, 첫 번째 문장만 Truncate 한다. 반대로, "only_second"인 경우 두 번째 문장만 truncate 한다. False로 지정한 경우 trauncate을 하지 않는다.

- max_length : 모델이 입력으로 받을 최대의 입력 길이. 별도로 지정하지 않은 경우 모델의 config에 맞게 지정된다.

- is_split_into_words (False) : 입력이 이미 tokenize가 되었는지 여부. True로 지정한 경우, 입력이 이미 tokenize가 되었다고 가정하고 별도의 Tokenize를 수행하지 않는다. 

- return_tensors : "tf" 면 tf.constant, "pt" 면 torch.Tensor, "np" 면 np.ndarray 로 결과를 리턴한다. 

- return_token_type_ids : token type id를 제공할지 여부. 기본적으로 제공함.

- return_attention_mask : attention mask를 제공할지 여부. 기본적으로 제공함

- return_special_tokens_mask (False) : special token mask를 리턴할지 여부

- return_length (False) : encode된 결과의 길이를 리턴할지 여부

- verbose (True) : 추가적인 출력이나 warning 등을 출력할지 여부 리턴 → BachEncoding

- input_ids : token is 의 리스트

- token_type_ids : token type id의 리스트

- attention_mask : 모델에 의해서 attention이 수행되어야 하는 인덱스

- num_truncated_tokens : truncate된 token의 개수 

- special_token_mask : 0인 경우 일반 Token, 1인 경우 Special token

- length : return_length=True 인 경우, encoding 된 출력의 길이
"""

output = tokenizer_bert(text=text, max_length=128, padding='max_length',return_tensors='pt',
                        return_token_type_ids=True, return_attention_mask=True)

input_ids = output.input_ids
attention_mask = output.attention_mask
token_type_ids = output.token_type_ids

print(input_ids)
print(attention_mask)
print(token_type_ids)

tensor([[    2, 35714,  8403,  6165,  8032,  8202, 25151,  8086,  2010,  4964,
          8033, 14194, 15729,  2011, 14396, 16980,  2070,     3,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

In [12]:
print(input_ids.shape)
print(attention_mask.shape)
print(token_type_ids.shape)

torch.Size([2, 128])
torch.Size([2, 128])
torch.Size([2, 128])


In [13]:
output = model_bert(input_ids, attention_mask, token_type_ids)
print(output.keys())

seq_output = output.last_hidden_state
pooler_output = output.pooler_output

print(seq_output.shape)
print(pooler_output.shape)

odict_keys(['last_hidden_state', 'pooler_output'])
torch.Size([2, 128, 768])
torch.Size([2, 768])


In [14]:
input_ids = torch.LongTensor([[31, 51, 99], [15, 5, 0]])
attention_mask = torch.LongTensor([[1, 1, 1], [1, 1, 0]])
token_type_ids = torch.LongTensor([[0, 0, 1], [0, 1, 0]])

input_data = {'input_ids':input_ids, 'attention_mask':attention_mask, 'token_type_ids':token_type_ids}

output = model_bert(**input_data)
print(output.keys())


odict_keys(['last_hidden_state', 'pooler_output'])


In [7]:
torch.nn.functional.one_hot(torch.tensor(0), num_classes=6).float()

tensor([1, 0, 0, 0, 0, 0])

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizerFast

class MyDataset(Dataset):
    def __init__(self, x_data, y_data, max_length=128, padding='max_length', num_classes=6):
        super(MyDataset, self).__init__()
        self.sentence = x_data
        self.sentiment = y_data
        self.num_classes = num_classes

        self.tokenizer = BertTokenizerFast.from_pretrained("kykim/bert-kor-base")

        self.max_length = max_length
        self.padding = padding
        self.return_tensors = 'pt'
        self.return_token_type_ids = True
        self.return_attention_mask = True

    def __len__(self):
        return len(self.sentence)

    def __getitem__(self, idx):
        ## sentence Tokenize ##
        x = self.sentence.iloc[idx][0]
        tokenizer_output = self.tokenizer(x, max_length=self.max_length, padding=self.padding,
                                          return_tensors=self.return_tensors,
                                          return_token_type_ids=self.return_token_type_ids,
                                          return_attention_mask=self.return_attention_mask)

        input_ids = tokenizer_output.input_ids
        attention_mask = tokenizer_output.attention_mask
        token_type_ids = tokenizer_output.token_type_ids

        ## sentiment ##
        y = torch.tensor(self.sentiment.iloc[idx][0])
        y = F.one_hot(y, num_classes = self.num_classes).float()

        return (input_ids[0], attention_mask[0], token_type_ids[0]), y

    def show_item(self, idx=0):
        feature, label = self.__getitem__(idx)

        print("input_ids's Shape : {}".format(feature[0].shape))
        print("attention_mask's Shape : {}".format(feature[1].shape))
        print("token_type_ids's Shape : {}".format(feature[2].shape))
        print("Label's Shape : {}".format(label.shape))

        return feature, label

In [4]:
def label2int(data, label_tags):
    for i in range(len(data)):
        data.iloc[i,1] = label_tags.index(data.iloc[i, 1])
    return data

In [5]:
    # label_tags
    label_tags = ['불안', '슬픔', '상처', '당황', '분노', '기쁨']

    train_path = "train.csv"
    test_path = "test.csv"

    train_data = pd.read_csv(train_path, encoding='cp949')
    test_data = pd.read_csv(test_path, encoding='cp949')


In [6]:
train_data = label2int(train_data, label_tags)
test_data = label2int(test_data, label_tags)

In [7]:
train_data

,sentence,label
0,오늘은 누구랑 같이 밥을 먹어야 할까?,0
1,난 남들 다 받는 과외를 못 받아서 공부를 못하는 것 같아.,1
2,오늘 친구가 몸이 너무 안 좋아서 친구 잘못으로 내가 대신 혼났어.,2
3,우리 주위에는 불치병으로 인하여 어려운 분들이 많은 것 같아.,2
4,아이들이 사춘기가 되니 예전처럼 나에게 먼저 말을 걸지 않고 자꾸 나를 피하려고 해.,3
...,...,...
9895,나 점차 초조해.,0
9896,아들을 특수부대에 보냈는데 부상을 당할까 봐 걱정돼.,0
9897,최근 들어 고혈압이 심해졌어. 뇌졸중까지 오는 게 당연한 거 같아서 슬퍼.,1
9898,대리님이 나한테 갑자기 프레젠테이션을 떠넘기셨는데 솔직히 너무 무책임하신 것 같아.,4


In [8]:
# your Data Pre-Processing
train_x, train_y = train_data.iloc[:, :1], train_data.iloc[:, 1:]
test_x, test_y = test_data.iloc[:, :1], test_data.iloc[:, 1:]

In [9]:
from sklearn.model_selection import train_test_split
train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y, stratify=train_y, random_state=17, test_size=0.05)

In [10]:
train_x

,sentence
1897,운동을 혼자 다니니까 외로워.
1861,아빠는 내가 태어나자마자 떠나버렸대.
4573,아들이 입대했어.
4236,집이 잘 사는 게 부러워 사사건건 친구에게 시비를 걸었어.
9066,언제 죽을지 몰라 두려워.
...,...
1689,남편이 매일 등산을 하는 나를 보고 비아냥거려.
2723,텔레비전에 나오는 연예인들은 나와 동갑인데도 건강하고 활력이 넘쳐.
5623,동생이 학교폭력에 시달리고 있어.
2654,나는 정말 운이 좋아. 세상에서 제일 좋은 직장 선배를 만났거든.


In [11]:
# Check Train, Valid, Test Image's Shape
print("The Shape of Train Images: ", train_x.shape)
print("The Shape of Valid Images: ", valid_x.shape)
print("The Shape of Test Images: ", test_x.shape)

# Check Train, Valid Label's Shape
print("The Shape of Train Labels: ", train_y.shape)
print("The Shape of Valid Labels: ", valid_y.shape)
print("The Shape of Valid Labels: ", test_y.shape)

The Shape of Train Images:  (9405, 1)
The Shape of Valid Images:  (495, 1)
The Shape of Test Images:  (100, 1)
The Shape of Train Labels:  (9405, 1)
The Shape of Valid Labels:  (495, 1)
The Shape of Valid Labels:  (100, 1)


In [12]:
# Create Dataset and DataLoader
train_dataset = MyDataset(train_x, train_y)
valid_dataset = MyDataset(valid_x, valid_y)
test_dataset = MyDataset(test_x, test_y)

NameError: name 'MyDataset' is not defined

In [ ]:
train_dataset.show_item(0)

In [15]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [16]:
train_dataset.show_item(0)

input_ids's Shape : torch.Size([128])
attention_mask's Shape : torch.Size([128])
token_type_ids's Shape : torch.Size([128])
Label's Shape : torch.Size([6])


((tensor([    2, 17406, 14887, 14767, 14073, 34517,  8149,  2016,     3,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,   

In [21]:
import numpy as np
from transformers import AutoModel, AutoTokenizer

class BERTDataset(Dataset):
    def __init__(self, data, max_len):
        super(BERTDataset, self).__init__()
        self.data = data
        self.max_len = max_len
        self.tokenizer = AutoTokenizer.from_pretrained("klue/bert-base", use_fast = True)

        self.inputs = [self.convert_token([data.iloc[i]['sentence']]) for i in range(len(self.data))]
        self.label = [np.int32(data.iloc[i]['label']) for i in range(len(self.data))]

    def convert_token(self, data):
        token = self.tokenizer.encode(data[0])
        attention_mask = [1] * len(token) + [0] * (self.max_len - len(token))
        token = token + self.tokenizer.convert_tokens_to_ids(["[PAD]"] * (self.max_len - len(token)))
        return [np.int32(attention_mask), np.int32(token)]

    def __getitem__(self, idx):
        return (self.inputs[idx][0], self.inputs[idx][1]), self.label[idx]

    def __len__(self):
        return len(self.label)

In [22]:
t_data = pd.concat([train_x, train_y], axis=1)
v_data = pd.concat([valid_x, valid_y], axis=1)
t_data.head()

,sentence,label
1897,운동을 혼자 다니니까 외로워.,3
1861,아빠는 내가 태어나자마자 떠나버렸대.,2
4573,아들이 입대했어.,1
4236,집이 잘 사는 게 부러워 사사건건 친구에게 시비를 걸었어.,2
9066,언제 죽을지 몰라 두려워.,0


In [23]:
v_data.head()

,sentence,label
5592,나는 비위가 약하다는 것은 알고 있었는데 너무 힘드네.,4
3275,아들이 하는 행동을 보면 내가 원하는 대로 되지 않는 것 같아서 마음이 상해.,1
6106,나 너무 화가나. 친구가 내가 아끼는 노트를 찢어버린 거 있지?,4
7642,설을 앞두고 갑자기 입원하게 돼서 너무 외로워.,3
4843,아이들끼리 싸우는 것도 그렇고 따돌리는 것도 그렇고. 학교 다니면서 이렇게 혼란스러...,3


In [24]:
t_dataloader = DataLoader(BERTDataset(t_data, 128), batch_size=32, shuffle=True)
v_dataloader = DataLoader(BERTDataset(v_data, 128), batch_size=32, shuffle=True)

print(t_dataloader)

Downloading:   0%|          | 0.00/289 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/243k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/483k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [14]:
train_dataset.show_item(10)

NameError: name 'train_dataset' is not defined

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from transformers import BertModel, AlbertModel
from transformers import AutoModel, AutoTokenizer

class MyModel_1(nn.Module):
    def __init__(self):
        super(MyModel_1, self).__init__()

#         self.bert = AlbertModel.from_pretrained("kykim/albert-kor-base")
        self.bert = AutoModel.from_pretrained("klue/bert-base")
        self.bert.requires_grad = True

        self.fc1 = nn.Linear(768, 128)
        self.fc2 = nn.Linear(128, 6)

        self.drop = nn.Dropout(p=0.2)
        self.act_fn = nn.ReLU()

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids, attention_mask).pooler_output 

#         total_vector = bert_output.last_hidden_state        # (batch, 128, 768)
#         cls_vector = bert_output.pooler_output              # (batch, 768)

        x = self.fc1(x)
        x = self.act_fn(x)
#         x = self.drop(x)

        x = self.fc2(x)
        return x

In [26]:
def get_Model(class_name):
    try:
        Myclass = eval(class_name)()
        return Myclass
    except NameError as e:
        print("Class [{}] is not defined".format(class_name))

In [27]:
model_name = "MyModel_1"
model = MyModel_1()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
optimizer = torch.optim.AdamW(model.parameters())
criterion = torch.nn.CrossEntropyLoss()

Some weights of the model checkpoint at klue/bert-base were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [28]:
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm, notebook


def calc_acc(output, label):
    o_val, o_idx = torch.max(output, dim=-1)
#     l_val, l_idx = torch.max(label, dim=-1)
    return (o_idx == label).sum().item()


def train(model, device, optimizer, criterion, epochs, train_loader, valid_loader=None) -> dict:
    """
    :param model: your model
    :param device: your device(cuda or cpu)
    :param optimizer: your optimizer
    :param criterion: loss function
    :param epoch: train epochs
    :param train_loader: train dataset
    :param valid_loader: valid dataset
    :return: history dictionary that contains train_loss, train_acc, valid_loss, valid_acc as list
    """
    history = {
        'train_loss': [],
        'train_acc': [],
        'valid_loss': [],
        'valid_acc': []
    }
    model.to(device)
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = train_acc = 0

        # in notebook
        # pabr = notebook.tqdm(enumerate(train_loader), file=sys.stdout)

        # in interpreter
        pbar = tqdm(enumerate(train_loader), file=sys.stdout)
        for batch_idx, (data, target) in pbar:
            att_mask, input_ids = data[0].long().to(device), data[1].long().to(device)
            target = target.long().to(device)

            optimizer.zero_grad()
            output = model(input_ids, att_mask)
            loss = criterion(output, target)
            acc = calc_acc(output, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_acc += acc

            acc = train_acc / (batch_idx * train_loader.batch_size + len(data))
            pbar.set_postfix(epoch=f'{epoch}/{epochs}', loss='{:.6f}, acc={:.3f}'.format(loss, acc))
        pbar.close()

        train_loss = train_loss / len(train_loader)
        train_acc = train_acc / len(train_loader.dataset)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)

        if valid_loader is not None:
            valid_loss, valid_acc = evaluate(model, device, criterion, valid_loader)

            history['valid_loss'].append(valid_loss)
            history['valid_acc'].append(valid_acc)

    return history


def evaluate(model, device, criterion, data_loader):
    """
    :param model: your model
    :param device: your device(cuda or cpu)
    :param criterion: loss function
    :param data_loader: valid or test Datasets
    :return: (valid or test) loss and acc
    """
    model.eval()
    total_loss = total_acc = 0

    with torch.no_grad():
        # in notebook
        # pabr = notebook.tqdm(enumerate(valid_loader), file=sys.stdout)

        # in interpreter
        pbar = tqdm(enumerate(data_loader), file=sys.stdout)

        for batch_idx, (data, target) in pbar:
            att_mask, input_ids = data[0].long().to(device), data[1].long().to(device)
            target = target.long().to(device)

            output = model(input_ids, att_mask)
            loss = criterion(output, target)
            acc = calc_acc(output, target)

            total_loss += loss.item()
            total_acc += acc

            acc = total_acc / (batch_idx * data_loader.batch_size + len(data))
            pbar.set_postfix(loss='{:.6f}, acc={:.3f}'.format(loss, acc))
        pbar.close()

    total_loss = total_loss / len(data_loader)
    total_acc = total_acc / len(data_loader.dataset)

    return total_loss, total_acc

In [29]:
# train
print("============================= Train =============================")
history = train(model, device, optimizer, criterion, 10, t_dataloader, v_dataloader)

============================= Train =============================
193it [01:30,  2.13it/s, epoch=1/10, loss=1.792742, acc=0.167]


KeyboardInterrupt: 

In [42]:
device

'cuda'

In [43]:
'cuda' if torch.cuda.is_available() else 'cpu'

'cpu'